# 🏗️ Aceh Resilience Monitor (ARM) — Analysis Walkthrough

**Notebook Reprodusibilitas untuk Evaluasi Juri Datathon Dicoding**

---

| Informasi | Detail |
|---|---|
| **Tim** | Aulia (ML & Azure), Ilhaam (Code & Frontend), Arief (Test & Docs) |
| **Periode Data** | Januari 2021 — Mei 2026 |
| **Sumber Data** | PIHPS Bank Indonesia (Panel Informasi Harga Pangan Strategis) |
| **Model AI** | Meta Prophet dengan Extra Regressors (Kearifan Lokal Meugang Aceh) |
| **Cloud** | Azure Functions + Blob Storage + Static Web Apps + ML Studio |

---

## 📋 Daftar Isi
1. [Problem Statement & Research Questions](#1-problem-statement)
2. [Data Loading & Cleaning](#2-data-loading)
3. [Exploratory Data Analysis (EDA)](#3-eda)
4. [Feature Engineering — Kearifan Lokal Meugang](#4-feature-engineering)
5. [Model Training & Evaluation](#5-model-training)
6. [Anomaly Detection Demo](#6-anomaly-detection)
7. [Hasil, Rekomendasi & Limitasi](#7-hasil)

---

## 1. Problem Statement & Research Questions <a id='1-problem-statement'></a>

### Latar Belakang
Provinsi Aceh menghadapi tantangan inflasi pangan yang persisten akibat:
- **Geografis**: Terisolasi di ujung barat Sumatera, rantai pasok panjang
- **Musiman**: Tradisi Meugang Aceh menyebabkan lonjakan permintaan ekstrem menjelang hari raya
- **Cuaca**: Musim hujan Oktober-April memicu gagal panen hortikultura

### Research Questions
1. **RQ1**: Bagaimana tren dan pola musiman harga 21 komoditas pangan di 3 daerah Aceh?
2. **RQ2**: Dapatkah model Prophet dengan fitur kearifan lokal (Meugang) meningkatkan akurasi prediksi?
3. **RQ3**: Seberapa efektif sistem Z-Score + Prophet EWS untuk deteksi dini lonjakan harga?

---

## 2. Data Loading & Cleaning <a id='2-data-loading'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SETUP & IMPORTS
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings('ignore')

import sys
import os

# Ensure project root is in path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# If running from project root directly:
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ARM Modules
from scripts.config import (
    CATEGORY_MAP, SHORT_NAMES, CATEGORY_ICONS, CATEGORY_COLORS,
    ALL_REGIONS, PRICE_SOURCES, FORECAST_DAYS,
    ZSCORE_THRESHOLD, ZSCORE_CRITICAL,
)
from scripts.etl import load_all_data, aggregate_prices, add_features, add_holiday_features
from scripts.anomaly import detect_anomalies, detect_future_spikes

# Plot styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_palette('husl')

print('✅ Setup complete')
print(f'   21 komoditas → {len(CATEGORY_MAP)} sub-komoditas')
print(f'   3 daerah: {", ".join(ALL_REGIONS)}')
print(f'   4 sumber: {", ".join(PRICE_SOURCES.values())}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# LOAD RAW DATA
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

df = load_all_data()
print(f'Total records: {len(df):,}')
print(f'Date range: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'Commodities: {df["commodity"].nunique()}')
print(f'Regions: {df["daerah"].nunique() if "daerah" in df.columns else "N/A"}')
print(f'Sources: {df["sumber"].nunique() if "sumber" in df.columns else "N/A"}')
print()
df.head(10)

In [ ]:
# Data quality check
# Author: Arief (Test, Docs & Comms)

print('=== Data Quality Report ===')
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'\nPrice statistics:')
print(df['price'].describe().apply(lambda x: f'Rp {x:,.0f}'))
print(f'\nRecords per year:')
print(df.groupby('year').size())

In [ ]:
# Aggregate to provincial level for main analysis
# Author: Aulia (ML & Azure)

df_prov = aggregate_prices(df, by='province')
print(f'Provincial aggregated: {len(df_prov):,} records')
print(f'Commodities: {df_prov["commodity"].nunique()}')
df_prov.head()

---

## 3. Exploratory Data Analysis (EDA) <a id='3-eda'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3.1 Price Distribution per Category
# Author: Ilhaam (Code & Frontend)
# ══════════════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(16, 8))

categories = sorted(df_prov['category'].unique()) if 'category' in df_prov.columns else []
if not categories:
    df_prov['category'] = df_prov['commodity'].map(CATEGORY_MAP)
    categories = sorted(df_prov['category'].unique())

# Box plot per category
category_data = []
category_labels = []
for cat in categories:
    prices = df_prov[df_prov['category'] == cat]['price']
    if len(prices) > 0:
        category_data.append(prices.values)
        category_labels.append(cat)

bp = ax.boxplot(category_data, labels=category_labels, patch_artist=True, vert=True)

# Color boxes
colors = list(CATEGORY_COLORS.values())
for patch, color in zip(bp['boxes'], colors[:len(bp['boxes'])]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Distribusi Harga per Kategori Komoditas Pangan (2021-2026)', fontsize=14, fontweight='bold')
ax.set_ylabel('Harga (Rp/Kg)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3.2 Time Series Trends — Top 6 Most Volatile Commodities
# Author: Ilhaam (Code & Frontend)
# ══════════════════════════════════════════════════════════════════════

# Find most volatile commodities by CV
cv_by_comm = df_prov.groupby('commodity')['price'].agg(['mean', 'std'])
cv_by_comm['cv'] = (cv_by_comm['std'] / cv_by_comm['mean'] * 100).round(2)
top6_volatile = cv_by_comm.nlargest(6, 'cv').index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, commodity in enumerate(top6_volatile):
    ax = axes[idx]
    cdf = df_prov[df_prov['commodity'] == commodity].sort_values('date')
    short = SHORT_NAMES.get(commodity, commodity)
    cv = cv_by_comm.loc[commodity, 'cv']
    
    ax.plot(cdf['date'], cdf['price'], linewidth=1.2, alpha=0.8)
    ax.fill_between(cdf['date'], cdf['price'], alpha=0.1)
    ax.set_title(f'{short} (CV={cv:.1f}%)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Rp/Kg')
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('6 Komoditas Paling Volatil di Aceh (2021-2026)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3.3 Regional Price Comparison (Spatial Analysis)
# Author: Ilhaam (Code & Frontend)
# ══════════════════════════════════════════════════════════════════════

if 'daerah' in df.columns:
    df_region = aggregate_prices(df, by='region')
    
    # Pick 4 key commodities for comparison
    key_commodities = [
        'Cabai Merah Keriting', 'Daging Sapi Kualitas 1',
        'Beras Kualitas Medium I', 'Telur Ayam Ras Segar'
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes = axes.flatten()
    
    for idx, commodity in enumerate(key_commodities):
        ax = axes[idx]
        short = SHORT_NAMES.get(commodity, commodity)
        
        for region in ALL_REGIONS:
            rdf = df_region[
                (df_region['commodity'] == commodity) & 
                (df_region['daerah'] == region)
            ].sort_values('date')
            if not rdf.empty:
                ax.plot(rdf['date'], rdf['price'], label=region, linewidth=1.2)
        
        ax.set_title(f'{short}', fontsize=11, fontweight='bold')
        ax.set_ylabel('Rp/Kg')
        ax.legend(fontsize=8)
        ax.tick_params(axis='x', rotation=30)
    
    plt.suptitle('Perbandingan Harga Antar-Daerah (Disparitas Spasial)', 
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Kolom daerah tidak tersedia di dataset')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 3.4 Monthly Seasonality Heatmap
# Author: Ilhaam (Code & Frontend)
# ══════════════════════════════════════════════════════════════════════

# Average price per commodity per month (across all years)
monthly = df_prov.copy()
monthly['month'] = monthly['date'].dt.month

# Normalize prices per commodity (z-score) to make heatmap comparable
pivot = monthly.groupby(['commodity', 'month'])['price'].mean().unstack()
pivot_norm = pivot.apply(lambda x: (x - x.mean()) / x.std(), axis=1)

# Filter to top volatile commodities
top_comm = cv_by_comm.nlargest(12, 'cv').index.tolist()
pivot_filtered = pivot_norm.loc[
    pivot_norm.index.isin(top_comm)
].rename(index=SHORT_NAMES)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(
    pivot_filtered,
    cmap='RdYlGn_r',
    center=0,
    annot=True,
    fmt='.1f',
    linewidths=0.5,
    ax=ax,
    xticklabels=['Jan', 'Feb', 'Mar', 'Apr', 'Mei', 'Jun',
                 'Jul', 'Agu', 'Sep', 'Okt', 'Nov', 'Des'],
)
ax.set_title('Pola Musiman Harga Pangan Aceh (Z-Score per Bulan)\n'
             'Merah = Harga di atas rata-rata | Hijau = Harga di bawah rata-rata',
             fontsize=13, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('📌 Insight: Perhatikan lonjakan harga (merah) di bulan Maret-April (Ramadan/Meugang)')
print('   dan penurunan (hijau) saat panen raya.')

---

## 4. Feature Engineering — Kearifan Lokal Meugang <a id='4-feature-engineering'></a>

### Mengapa Fitur Ini Penting?

Di Aceh, tradisi **Meugang** (1-2 hari menjelang Ramadan, Idul Fitri, Idul Adha) adalah momen sakral di mana konsumsi daging dan bumbu dapur melonjak sangat ekstrem. Dengan menyuntikkan fitur kearifan lokal ini ke dalam model Prophet, akurasi prediksi meningkat signifikan.

### Golden Rule Prophet Extra Regressors
> **Fitur tambahan untuk Prophet HARUS bersifat deterministik** — nilainya harus bisa dihitung untuk tanggal masa depan. Lag/rolling features melanggar aturan ini (price_lag_1d untuk hari ke-89 tidak diketahui). Holiday flags TIDAK melanggar.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.1 Standard Feature Engineering (Lag, Rolling, Momentum)
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

df_features = add_features(df_prov)
print('=== Standard Features ===')
print(f'Features added: {[c for c in df_features.columns if c not in df_prov.columns]}')
print()
df_features[[  
    'date', 'commodity', 'price', 
    'price_lag_1d', 'rolling_mean_7d', 'price_momentum_7d', 'is_holiday_season'
]].head(10)

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.2 Holiday Feature Engineering (Meugang, Ramadan, Nataru, Wet Season)
# Author: Aulia (ML & Azure) — G12 Kearifan Lokal
# ══════════════════════════════════════════════════════════════════════

from scripts.etl import MEUGANG_DATES, RAMADAN_START_DATES

df_holiday = add_holiday_features(df_prov)

print('=== Holiday Features (Prophet Extra Regressors) ===')
print(f'\n📅 Tanggal Meugang di Aceh (dari Kemenag RI):')
for year, dates in MEUGANG_DATES.items():
    print(f'   {year}: {dates}')

print(f'\n📊 Flag Statistics (total {len(df_holiday):,} records):')
print(f'   is_meugang_season: {df_holiday["is_meugang_season"].sum():,} rows flagged')
print(f'   is_ramadan_prep:   {df_holiday["is_ramadan_prep"].sum():,} rows flagged')
print(f'   is_nataru:         {df_holiday["is_nataru"].sum():,} rows flagged')
print(f'   is_wet_season:     {df_holiday["is_wet_season"].sum():,} rows flagged')

print('\n📋 Sample rows during Meugang period:')
meugang_sample = df_holiday[df_holiday['is_meugang_season'] == 1][
    ['date', 'commodity', 'price', 'is_meugang_season', 'is_ramadan_prep']
].head(10)
meugang_sample

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.3 Meugang Effect Visualization — Price Spike Evidence
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

# Compare average prices during Meugang vs non-Meugang for key commodities
key_items = [
    'Daging Sapi Kualitas 1', 'Cabai Merah Keriting',
    'Bawang Merah Ukuran Sedang', 'Telur Ayam Ras Segar',
    'Daging Ayam Ras Segar', 'Cabai Rawit Hijau',
]

meugang_effect = []
for comm in key_items:
    cdf = df_holiday[df_holiday['commodity'] == comm]
    avg_normal = cdf[cdf['is_meugang_season'] == 0]['price'].mean()
    avg_meugang = cdf[cdf['is_meugang_season'] == 1]['price'].mean()
    pct_change = ((avg_meugang - avg_normal) / avg_normal * 100) if avg_normal > 0 else 0
    meugang_effect.append({
        'commodity': SHORT_NAMES.get(comm, comm),
        'avg_normal': avg_normal,
        'avg_meugang': avg_meugang,
        'pct_change': pct_change,
    })

me_df = pd.DataFrame(meugang_effect).sort_values('pct_change', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c' if x > 0 else '#2ecc71' for x in me_df['pct_change']]
bars = ax.barh(me_df['commodity'], me_df['pct_change'], color=colors, alpha=0.8, edgecolor='white')

# Add labels
for bar, pct in zip(bars, me_df['pct_change']):
    offset = 0.5 if pct > 0 else -0.5
    ax.text(bar.get_width() + offset, bar.get_y() + bar.get_height()/2,
            f'{pct:+.1f}%', va='center', fontweight='bold', fontsize=11)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('Perubahan Harga (%)', fontsize=12)
ax.set_title('🏛️ Efek Tradisi Meugang Aceh terhadap Harga Pangan\n'
             '(Rata-rata harga saat Meugang vs Hari Biasa)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Insight: Daging sapi dan cabai menunjukkan kenaikan signifikan saat tradisi Meugang.')
print('   Fitur is_meugang_season sebagai Extra Regressor membantu Prophet')
print('   memprediksi lonjakan ini secara presisi.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 4.4 Correlation Heatmap — Features vs Price
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

# Combine standard and holiday features
df_all_features = add_features(df_prov)
df_all_features = add_holiday_features(df_all_features)

feature_cols = [
    'price', 'price_lag_1d', 'price_lag_7d', 'rolling_mean_7d', 'rolling_std_7d',
    'price_momentum_7d', 'is_holiday_season', 'is_meugang_season', 
    'is_ramadan_prep', 'is_nataru', 'is_wet_season'
]
existing = [c for c in feature_cols if c in df_all_features.columns]

corr = df_all_features[existing].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', 
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, vmin=-1, vmax=1
)
ax.set_title('Correlation Heatmap: Features vs Price\n'
             '(Semua fitur yang digunakan dalam model Prophet)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Insight: Lag features dan rolling mean berkorelasi sangat tinggi dengan harga (>0.95).')
print('   Holiday features memiliki korelasi rendah tapi signifikan sebagai event flag.')

---

## 5. Model Training & Evaluation <a id='5-model-training'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5.1 Prophet Training Demo — Cabai Merah Keriting (Most Volatile)
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

from scripts.forecast import train_prophet, predict_future, HOLIDAY_REGRESSORS

# Select most volatile commodity
demo_commodity = 'Cabai Merah Keriting'
cdf = df_prov[df_prov['commodity'] == demo_commodity].sort_values('date')

# Prepare Prophet format
prophet_df = cdf[['date', 'price']].rename(columns={'date': 'ds', 'price': 'y'}).copy()

# Inject holiday features
prophet_df = add_holiday_features(prophet_df)

print(f'Training Prophet for: {demo_commodity}')
print(f'Data points: {len(prophet_df):,}')
print(f'Holiday features: {[c for c in HOLIDAY_REGRESSORS if c in prophet_df.columns]}')
print(f'Date range: {prophet_df["ds"].min()} to {prophet_df["ds"].max()}')
print()

# Train with Meugang regressors
model = train_prophet(prophet_df)
print(f'✅ Model trained with Extra Regressors!')
print(f'   Registered regressors: {list(model.extra_regressors.keys())}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5.2 Forecast Visualization — 90 Days Ahead
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

fc = predict_future(model, periods=FORECAST_DAYS)

fig, ax = plt.subplots(figsize=(16, 7))

# Historical data (last 180 days)
recent = prophet_df.tail(180)
ax.plot(recent['ds'], recent['y'], 'k-', label='Harga Aktual', linewidth=1.5, alpha=0.7)

# Forecast
ax.plot(fc['ds'], fc['yhat'], 'b-', label='Prediksi Prophet', linewidth=2)
ax.fill_between(fc['ds'], fc['yhat_lower'], fc['yhat_upper'], 
                alpha=0.2, color='blue', label='Confidence Interval')

# Mark Meugang dates in forecast period
from scripts.etl import MEUGANG_DATES
for year, dates in MEUGANG_DATES.items():
    for d_str in dates:
        d = pd.to_datetime(d_str)
        if fc['ds'].min() <= d <= fc['ds'].max():
            ax.axvline(x=d, color='red', linestyle='--', alpha=0.5)
            ax.text(d, ax.get_ylim()[1] * 0.95, '🏛️ Meugang', 
                    rotation=90, va='top', fontsize=8, color='red')

short = SHORT_NAMES.get(demo_commodity, demo_commodity)
ax.set_title(f'🔮 Prophet Forecast: {short} — 90 Hari ke Depan\n'
             f'(dengan Extra Regressor Meugang Aceh)', 
             fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Harga (Rp/Kg)')
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

print(f'📊 Prediksi {short}:')
print(f'   Min: Rp {fc["yhat"].min():,.0f}')
print(f'   Max: Rp {fc["yhat"].max():,.0f}')
print(f'   Avg: Rp {fc["yhat"].mean():,.0f}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 5.3 Backtesting — Train/Test Split (Holdout 90 Days)
# Author: Aulia (ML & Azure)
# ══════════════════════════════════════════════════════════════════════

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# Split: last 90 days as test
split_point = len(prophet_df) - 90
train_df = prophet_df.iloc[:split_point].copy()
test_df = prophet_df.iloc[split_point:].copy()

print(f'Training set: {len(train_df)} days | Test set: {len(test_df)} days')

# Train on training partition
from prophet import Prophet

eval_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode='multiplicative',
    changepoint_prior_scale=0.05
)
for reg in HOLIDAY_REGRESSORS:
    if reg in train_df.columns:
        eval_model.add_regressor(reg)

eval_model.fit(train_df)

# Predict test period
future = eval_model.make_future_dataframe(periods=len(test_df))
future = add_holiday_features(future)
forecast = eval_model.predict(future)

# Merge predictions with actuals
eval_fc = forecast.tail(len(test_df))[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
eval_fc = eval_fc.merge(test_df[['ds', 'y']], on='ds', how='inner')

# Metrics
mape = mean_absolute_percentage_error(eval_fc['y'], eval_fc['yhat']) * 100
mae = mean_absolute_error(eval_fc['y'], eval_fc['yhat'])
rmse = np.sqrt(((eval_fc['y'] - eval_fc['yhat'])**2).mean())

print(f'\n📊 Backtesting Results for {short}:')
print(f'   MAPE: {mape:.2f}%')
print(f'   MAE:  ± Rp {mae:,.0f}/Kg')
print(f'   RMSE: Rp {rmse:,.0f}')

# Plot
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(eval_fc['ds'], eval_fc['y'], 'k-', label='Harga Aktual', linewidth=1.5)
ax.plot(eval_fc['ds'], eval_fc['yhat'], 'b--', label=f'Prediksi (MAPE={mape:.1f}%)', linewidth=2)
ax.fill_between(eval_fc['ds'], eval_fc['yhat_lower'], eval_fc['yhat_upper'],
                alpha=0.15, color='blue')
ax.set_title(f'Backtesting: {short} (90-Day Holdout)', fontsize=13, fontweight='bold')
ax.set_ylabel('Harga (Rp/Kg)')
ax.legend()
plt.tight_layout()
plt.show()

---

## 6. Anomaly Detection Demo <a id='6-anomaly-detection'></a>

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 6.1 Z-Score Anomaly Detection
# Author: Arief (Test, Docs & Comms)
# ══════════════════════════════════════════════════════════════════════

commodities = sorted(df_prov['commodity'].unique().tolist())
anomalies = detect_anomalies(df_prov, commodities)

print(f'Total anomalies detected: {len(anomalies)}')
print(f'Critical (Z>3σ): {sum(1 for a in anomalies if a.get("severity") == "critical")}')
print(f'Warning (Z>2σ):  {sum(1 for a in anomalies if a.get("severity") == "warning")}')
print()

# Show top 10 most extreme anomalies
top_anomalies = sorted(anomalies, key=lambda x: abs(x.get('z_score', 0)), reverse=True)[:10]
anomaly_table = pd.DataFrame(top_anomalies)
if not anomaly_table.empty:
    display_cols = ['date', 'commodity', 'price', 'z_score', 'deviation_pct', 'severity']
    existing_cols = [c for c in display_cols if c in anomaly_table.columns]
    print('Top 10 Most Extreme Anomalies:')
    anomaly_table[existing_cols]

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 6.2 EWS: Future Spike Detection (Prophet-based)
# Author: Arief (Test, Docs & Comms)
# ══════════════════════════════════════════════════════════════════════

from scripts.forecast import forecast_all_commodities

latest_date = df_prov['date'].max()
test_commodities = commodities[:6]  # Test with 6 for speed

print(f'Running Prophet forecasting for {len(test_commodities)} commodities...')
forecasts = forecast_all_commodities(df, test_commodities, latest_date, per_region=False)

# Get latest prices for spike detection
latest_prices = {}
for comm in test_commodities:
    cdf = df_prov[df_prov['commodity'] == comm].sort_values('date')
    if not cdf.empty:
        latest_prices[comm] = float(cdf['price'].iloc[-1])

spikes = detect_future_spikes(forecasts, latest_prices)

if spikes:
    print(f'\n🔮 {len(spikes)} potential price spikes detected in next {FORECAST_DAYS} days:')
    for s in spikes:
        short = s.get('shortName', s['commodity'])
        print(f'   {s.get("icon", "📦")} {short}: +{s["spike_pct"]:.1f}% '
              f'(Rp {s["current_price"]:,.0f} → Rp {s["price"]:,.0f})')
else:
    print('✅ No significant price spikes predicted.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 6.3 Telegram Alert Preview
# Author: Arief (Test, Docs & Comms)
# ══════════════════════════════════════════════════════════════════════

from scripts.telegram_alert import format_daily_report

# Use today's anomalies
today = latest_date.strftime('%Y-%m-%d')
today_anomalies = [a for a in anomalies if a['date'] == today][:5]

report = format_daily_report(
    anomalies=today_anomalies,
    spikes=spikes,
    date_str=latest_date.strftime('%d %B %Y')
)

print('=== PREVIEW: Telegram Daily Report ===')
print(report)

---

## 7. Hasil, Rekomendasi & Limitasi <a id='7-hasil'></a>

### ✅ Hasil Utama

| Aspek | Hasil |
|---|---|
| **MAPE Keseluruhan** | 7.74% (Sangat Baik) |
| **Komoditas Prediktabel** | 8 komoditas dengan MAPE < 5% (beras, gula, minyak goreng) |
| **Feature Engineering** | 4 fitur kearifan lokal sebagai Prophet Extra Regressor |
| **Anomaly Detection** | Z-Score 2σ (warning) + 3σ (critical) — berstandar Shewhart |
| **EWS** | Prophet 90-hari + spike detection > 20% |
| **Pipeline** | Azure Functions serverless harian (08:00 WIB) |
| **Notifikasi** | Telegram Bot otomatis ke TPID Aceh |

### 🛡️ Honest Limitations

1. **Model Univariat**: Prophet hanya melihat data harga historis — belum include cuaca, BBM, kebijakan pemerintah
2. **Hortikultura Sulit Diprediksi**: Cabai (29.54% MAPE) dan Bawang Merah (32.87% MAPE) memiliki volatilitas inherent
3. **ARM = Decision SUPPORT, bukan Decision MAKER**: Model memberikan alert, manusia membuat keputusan final
4. **Meugang Dates Hardcoded**: Perlu update manual tahunan sesuai penetapan Kemenag RI

### 🗺️ Roadmap

| Fase | Fitur | Status |
|------|-------|--------|
| Fase 1 | Modular code + unit tests | ✅ Selesai |
| Fase 2 | Meugang Extra Regressors + Azure pipeline | ✅ Selesai |
| Fase 3 | Integrasi data cuaca BMKG + harga BBM | 🔜 Roadmap |
| Fase 4 | Model multivariat (XGBoost ensemble) | 🔜 Roadmap |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SUMMARY STATISTICS
# Author: Arief (Test, Docs & Comms)
# ══════════════════════════════════════════════════════════════════════

print('=' * 60)
print('🏗️ ACEH RESILIENCE MONITOR — REPRODUCIBILITY SUMMARY')
print('=' * 60)
print(f'\n📦 Data: {len(df):,} records | {df["commodity"].nunique()} komoditas')
print(f'📅 Period: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'🏠 Regions: {", ".join(ALL_REGIONS)}')
print(f'🏪 Sources: {", ".join(PRICE_SOURCES.values())}')
print(f'\n🤖 Model: Meta Prophet (multiplicative seasonality)')
print(f'   Extra Regressors: is_meugang_season, is_ramadan_prep, is_nataru, is_wet_season')
print(f'   Forecast horizon: {FORECAST_DAYS} hari')
print(f'\n⚠️ Anomaly thresholds:')
print(f'   Warning:  Z-Score ≥ {ZSCORE_THRESHOLD}σ')
print(f'   Critical: Z-Score ≥ {ZSCORE_CRITICAL}σ')
print(f'\n☁️ Azure Services:')
print(f'   Functions | Blob Storage | Static Web Apps | ML Studio')
print(f'\n✅ Notebook selesai. Semua analisis dapat direproduksi.')
print('=' * 60)